# Image Segmentation in 2D Images: Computer Vision 102, Going from Zero to Hero

Welcome to the pixel-level frontier of Machine Learning. In this tutorial, we will implement **Semantic Segmentation**. 

Unlike standard Convolutional Neural Networks (CNNs) that shrink an image down to a single vector of classes using Pooling and Dense layers, a Segmentation network uses an **Encoder-Decoder** architecture (often utilizing Atrous/Dilated convolutions). 
1. **The Encoder** shrinks the image to extract deep features and context.
2. **The Decoder** upsamples those features back to the original image resolution $H \times W$ to assign a class to every pixel.

### Our Analytical Pipeline:
1. **Environment Setup:** Importing the deep learning tensor libraries.
2. **Model Instantiation:** Loading a pre-trained **DeepLabV3** model.
3. **Data Ingestion & Preprocessing:** Formatting a standard image into a normalized mathematical tensor.
4. **Inference (The Forward Pass):** Pushing the tensor through the network.
5. **Post-Processing:** Mathematically converting the raw output logits back into a human-readable RGB mask.

In [ ]:
# Cell 2: Environment Setup and Imports
# We use PyTorch and Torchvision, the industry standards for advanced Computer Vision research.
import torch
import torchvision.transforms as T
from torchvision import models
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import urllib.request

# Verify PyTorch is working and check for GPU acceleration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"PyTorch Version: {torch.__version__}")

## Step 1: Loading the Architecture (DeepLabV3)

Training a segmentation model from scratch requires massive datasets (like MS COCO or Pascal VOC) and days of GPU time. Instead, we use **Transfer Learning**.

We will load **DeepLabV3** with a **ResNet-50** backbone. 
DeepLab is famous for its analytically powerful *Atrous Spatial Pyramid Pooling (ASPP)*. Instead of just standard convolutions, it uses "dilated" filters that expand the network's field of view without increasing the number of parameters or losing spatial resolution.

In [ ]:
# Cell 4: Model Instantiation

# Load the pre-trained DeepLabV3 model. 
# weights='DEFAULT' loads the best available weights trained on the COCO dataset (21 classes).
model = models.segmentation.deeplabv3_resnet50(weights='DEFAULT')

# Move the model to the appropriate device (CPU/GPU)
model = model.to(device)

# Set the model to evaluation mode. 
# This mathematically disables Dropout and Batch Normalization updates, ensuring deterministic predictions.
model.eval()

print("DeepLabV3 model successfully loaded and set to evaluation mode.")

## Step 2: Data Ingestion and Preprocessing

Neural networks do not understand JPEG files; they understand Tensors. Furthermore, pre-trained models expect input data to have the exact same mathematical distribution as the data they were trained on.

We must define a transformation pipeline to:
1. Resize the image (optional but good for memory limits).
2. Convert the PIL Image to a PyTorch Tensor with shape $[C, H, W]$ (Channels, Height, Width).
3. **Normalize** the tensor using the ImageNet dataset's exact mean ($\mu$) and standard deviation ($\sigma$):
   $$x_{normalized} = \frac{x - \mu}{\sigma}$$

In [ ]:
# Cell 6: Transformation Pipeline and Image Loading

# 1. Define the exact transformations required by the pre-trained model
preprocess = T.Compose([
    T.ToTensor(), # Converts pixel range [0, 255] to [0.0, 1.0] and shape to [C, H, W]
    T.Normalize(mean=[0.485, 0.456, 0.406], # Standard ImageNet means
                std=[0.229, 0.224, 0.225])  # Standard ImageNet standard deviations
])

# 2. Download a sample image for our tutorial (A classic street scene)
url = "https://pytorch.org/tutorials/_static/img/tv_tutorial/tv_image05.png"
image_path = "sample_street.png"
urllib.request.urlretrieve(url, image_path)

# 3. Load the image using Pillow (PIL)
input_image = Image.open(image_path).convert("RGB")

# 4. Apply transformations
input_tensor = preprocess(input_image)

# 5. Create a mini-batch of size 1. 
# PyTorch models expect batches, so we add an extra dimension at the front.
# Shape becomes [1, C, H, W] (Batch, Channels, Height, Width)
input_batch = input_tensor.unsqueeze(0).to(device)

print(f"Original Image Size: {input_image.size}")
print(f"Tensor Shape for Model: {input_batch.shape}")

## Step 3: Inference and Analytical Post-Processing

When we pass the image through the model, it doesn't return colors. It returns an unnormalized mathematical tensor called **logits**. 

For a 21-class model, the output shape will be $[1, 21, H, W]$. 
This means for every single pixel, the model outputs 21 numbers representing the confidence for each of the 21 classes (Background, Person, Car, Dog, etc.).

To find the final prediction for a pixel, we apply the `argmax` function across the class dimension (dimension 1). 
$$Output_{H, W} = \text{argmax}(Logits_{21, H, W})$$
This collapses the 21 probabilities into a single integer representing the winning class ID, resulting in a 2D matrix of shape $[H, W]$.

In [ ]:
# Cell 8: The Forward Pass and Argmax

# Disable gradient calculation to save memory and compute (we are not training)
with torch.no_grad():
    # Push the tensor through the model
    output = model(input_batch)['out'][0] # Extract the main output from the dict

print(f"Raw Output Logits Shape: {output.shape}") 
# Expected: torch.Size([21, Height, Width])

# Apply argmax across the 21 classes (dimension 0 of the squeezed output tensor)
# This gives us the predicted class index (0-20) for every single pixel
predicted_mask = output.argmax(dim=0)

print(f"2D Mask Shape after Argmax: {predicted_mask.shape}") 
# Expected: torch.Size([Height, Width])

## Step 4: Decoding and Visualization

We now have a 2D matrix filled with integers (e.g., 0 for background, 15 for person, 7 for car). To make this human-readable, we must map these mathematical integers back into specific RGB colors.

We will create a color palette and apply it to our 2D mask, then plot it side-by-side with the original image to evaluate the model's performance analytically.

In [ ]:
# Cell 10: Decoding the Matrix to RGB

def decode_segmap(image_tensor, num_classes=21):
    """
    Maps a 2D tensor of class integers to a 3D tensor of RGB colors.
    """
    # Convert the PyTorch tensor to a 2D Numpy Array
    image_numpy = image_tensor.cpu().numpy()
    
    # Initialize empty RGB channels
    r = np.zeros_like(image_numpy).astype(np.uint8)
    g = np.zeros_like(image_numpy).astype(np.uint8)
    b = np.zeros_like(image_numpy).astype(np.uint8)
    
    # Define a standard color map for PASCAL VOC classes
    colors = np.array([
        [0, 0, 0], [128, 0, 0], [0, 128, 0], [128, 128, 0], [0, 0, 128], 
        [128, 0, 128], [0, 128, 128], [128, 128, 128], [64, 0, 0], [192, 0, 0], 
        [64, 128, 0], [192, 128, 0], [64, 0, 128], [192, 0, 128], [64, 128, 128], 
        [192, 128, 128], [0, 64, 0], [128, 64, 0], [0, 192, 0], [128, 192, 0], 
        [0, 64, 128]
    ])
    
    # Map the class integers to the colors
    for l in range(0, num_classes):
        idx = image_numpy == l
        r[idx] = colors[l, 0]
        g[idx] = colors[l, 1]
        b[idx] = colors[l, 2]
        
    # Stack the channels back together into a (H, W, 3) image
    rgb_mask = np.stack([r, g, b], axis=2)
    return rgb_mask

# Decode our predicted mask
rgb_segmentation_map = decode_segmap(predicted_mask)

# Visualize the results side-by-side
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

axes[0].imshow(input_image)
axes[0].set_title("Original Input Image", fontsize=16)
axes[0].axis('off')

axes[1].imshow(rgb_segmentation_map)
axes[1].set_title("Semantic Segmentation Mask", fontsize=16)
axes[1].axis('off')

plt.tight_layout()
plt.show()

## Conclusion

You have successfully written an end-to-end Semantic Segmentation pipeline! Let's review the analytical workflow:

1. **The mathematical hurdle:** You cannot just classify an image. You must maintain spatial resolution throughout the network.
2. **The Tensor Transformation:** We standardized our data to have a mean of 0 and a variance aligned with ImageNet ($X_{norm}$).
3. **The Logits:** The network output a 3D tensor of shape $C \times H \times W$, representing the confidence score of every class for every pixel.
4. **The Collapse:** By applying `argmax`, we flattened the probability distributions into absolute decisions, leaving us with the final blueprint of the image's layout.

This precise, pixel-level understanding is the exact technology used today to navigate self-driving cars, detect tumors in medical MRIs, and create the background-blur effect on your video calls.